In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Perceptron
import json


In [3]:
# Generate a synthetic dataset with 50 input features
# Let's create 1000 samples with 50 features each and binary labels
np.random.seed(42)  # For reproducibility

X = np.random.rand(1000, 50)  # 1000 samples, 50 features
y = np.random.randint(2, size=(1000, 1))  # Binary labels (0 or 1)

print(X.shape, y.shape)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# print(y_train.shape, y_test.shape)
# print(X_train.shape, X_test.shape)


(1000, 50) (1000, 1)


In [4]:
# Define a single-layer perceptron model
model = Sequential([
    Dense(1, input_dim=50, activation='relu'),  # One neuron, 50 inputs, sigmoid activation
    # BatchNormalization()
])

# Compile the model
model.compile(optimizer='sgd',  # Stochastic Gradient Descent
              loss='binary_crossentropy',  # Loss function for binary classification
              metrics=['accuracy'])


# Train the model
model.fit(X_train, y_train, epochs=20, batch_size=10, verbose=1)


Epoch 1/20


/home/nestor/.local/lib/python3.10/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1737766355.250882    7977 service.cc:145] XLA service 0x7f3438004510 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1737766355.250905    7977 service.cc:153]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.4996 - loss: 5.7061  
Epoch 2/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 284us/step - accuracy: 0.5331 - loss: 7.4439
Epoch 3/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 282us/step - accuracy: 0.4938 - loss: 8.0699
Epoch 4/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 286us/step - accuracy: 0.5113 - loss: 7.7907
Epoch 5/20


I0000 00:00:1737766355.516439    7977 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 296us/step - accuracy: 0.5438 - loss: 7.2733
Epoch 6/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 279us/step - accuracy: 0.5184 - loss: 7.6780
Epoch 7/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 269us/step - accuracy: 0.5151 - loss: 7.7308
Epoch 8/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 276us/step - accuracy: 0.5317 - loss: 7.4666
Epoch 9/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 284us/step - accuracy: 0.5270 - loss: 7.5409
Epoch 10/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 422us/step - accuracy: 0.5231 - loss: 7.6036
Epoch 11/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 369us/step - accuracy: 0.5364 - loss: 7.3912
Epoch 12/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 433us/step - accuracy: 0.5272 - loss: 7.5377
Epoch 13/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 413us/step - accuracy: 0.5185 - loss: 7.6766
Epoch 14/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 659us/step - accuracy: 0.4869 - loss: 8.1802
Epoch 15/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 463us/step - accuracy: 0.4736 - loss: 8.3917
Epoch 16/20
80/80 ━━━━━━━━━━━━━━━━━━━━ 0s 381us/step - accu

In [5]:
# Evaluate the model on the test set
loss, accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {loss:.4f}')
print(f'Test Accuracy: {accuracy:.4f}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4660 - loss: 8.5134 
Test Loss: 8.1306
Test Accuracy: 0.4900


In [6]:
# Access the weights
weights, bias = model.get_weights()

# Save the weights to a JSON file
weights_dict = {
    "weights": weights.tolist(),
    "bias": bias.tolist()
}

# print(weights)

In [7]:
with open("perceptron_weights.json", "w") as fw:
    json.dump(weights_dict, fw)

print("Weights and bias saved to perceptron_weights.json")

Weights and bias saved to perceptron_weights.json


In [8]:
# Read the JSON file
with open('perceptron_weights.json', 'r') as fr:
    data = json.load(fr)

# Convert the JSON data to a format suitable for Verilog
with open('weights_values.mem', 'w') as fmem:
    for weight_value in data['weights']:
        weight_valueQ15 = weight_value[0] * 2**15 # Convert to Q15.0 fixed-point format
        weight_valueQ15 = int(weight_valueQ15) # Convert to integer
        # Get the raw binary representation
        binary_representation = bin(weight_valueQ15 & 0xFFFF)[2:].zfill(16)
        fmem.write(f"{binary_representation}\n")
        # print(weight_valueQ15)    
        
    bias_value = data['bias']
    bias_valueQ15 = bias_value[0] * 2**15 # Convert to Q15.0 fixed-point format
    bias_valueQ15 = int(bias_valueQ15) # Convert to integer
    # Get the raw binary representation
    binary_representation = bin(bias_valueQ15 & 0xFFFF)[2:].zfill(16)
    fmem.write(f"{binary_representation}\n") 
    

# Save iput data to a file
XQ15 = X * 2**15
XQ15 = XQ15.astype(np.int16)

with open('input_values.mem', 'w') as fmem:
    for value in XQ15[0]:
        binary_representation = bin(value & 0xFFFF)[2:].zfill(16)
        fmem.write(f"{binary_representation}\n")  # Convert value to float before formatting as binary
# print(XQ15)

In [9]:
# Example single input (make sure it has the correct shape)
single_input = X[0].reshape(1, -1)

# Print the input value
# print("Input value for the single input:", single_input)

# Make a prediction
prediction = model.predict(single_input)

predictionQ15 = prediction[0][0] * 2**15 # Convert to Q15.0 fixed-point format

# Print the prediction and the classified class
print("Prediction for the single input:", prediction)
print("Prediction for the single input in Q15.0 format:", int(predictionQ15))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
Prediction for the single input: [[35.564594]]
Prediction for the single input in Q15.0 format: 1165380


In [10]:
# Assuming single_input and weights are already defined as numpy arrays
# Slice the first ten elements
single_input_ = single_input[0]
weights_ = weights[0:50]

print(single_input_)
print(weights_)

# Perform the dot product with the first ten elements
product = np.dot(single_input_, weights_)
print("Product:", product)
print(product +bias_value[0])

[0.37454012 0.95071431 0.73199394 0.59865848 0.15601864 0.15599452
 0.05808361 0.86617615 0.60111501 0.70807258 0.02058449 0.96990985
 0.83244264 0.21233911 0.18182497 0.18340451 0.30424224 0.52475643
 0.43194502 0.29122914 0.61185289 0.13949386 0.29214465 0.36636184
 0.45606998 0.78517596 0.19967378 0.51423444 0.59241457 0.04645041
 0.60754485 0.17052412 0.06505159 0.94888554 0.96563203 0.80839735
 0.30461377 0.09767211 0.68423303 0.44015249 0.12203823 0.49517691
 0.03438852 0.9093204  0.25877998 0.66252228 0.31171108 0.52006802
 0.54671028 0.18485446]
[[ 2.860667  ]
 [ 2.5620556 ]
 [ 1.0446972 ]
 [ 2.599     ]
 [ 0.7691692 ]
 [ 1.3555586 ]
 [ 0.4939922 ]
 [ 0.32522404]
 [ 2.3006027 ]
 [ 1.0422493 ]
 [ 0.72542906]
 [ 0.68656975]
 [ 2.1170354 ]
 [-0.02437824]
 [ 2.0274417 ]
 [ 0.18439764]
 [ 0.43653882]
 [ 1.0746144 ]
 [ 1.5065233 ]
 [ 0.7790691 ]
 [ 0.8075274 ]
 [ 2.5714295 ]
 [ 1.6443698 ]
 [ 0.22912388]
 [ 1.2134953 ]
 [ 2.379751  ]
 [ 1.1203907 ]
 [ 0.2683588 ]
 [ 2.1422892 ]
 [ 1.

In [11]:
# Define the file path
file_path = 'ringer_params.txt'

# Initialize an empty list to store the vector
vector = []

# Open the file and read the lines
with open(file_path, 'r') as file:
    lines = file.readlines()
    # Convert each line to a float and store it in the vector
    vector = [float(line.strip()) for line in lines]

# Print the vector
vector_np = np.array(vector)
print(vector_np)


[ 1.6764456  -1.53755701 -2.30328941 -1.63798594  1.85131466  0.37882406
 -0.328742   -0.22401194 -0.618577    0.49023652 -0.42926419 -0.42166159
  0.26507398 -0.30865371 -0.09595657  0.12437478 -0.14020289  0.16238448
 -0.38597462 -0.24609151  2.01618648 -2.1321311  -2.12202001 -2.16079259
  2.22524405  1.64988542 -1.84096849 -1.82588863 -1.62897825  2.09816194
 -0.0479631  -0.19153105  0.1740219   0.5022198  -0.26825178 -1.85738361
  1.726982    2.13008118  1.7760576  -1.59548855 -3.67750788  3.31737018
  3.52869725  2.91090298 -3.76481485 -3.95939612  2.9136765   3.3774159
  2.99220681 -4.24854708 -3.84060168  2.91511869  3.3037231   2.48241854
 -4.18867874 -3.79197216  2.33185935  3.24598193  2.16058826 -3.85004568
 -3.9009552   2.13865304  2.76338363  2.08198309 -3.98795652 -3.47019935
  1.69065416  2.06341028  1.72047353 -3.58478713 -3.08765268  1.77980638
  2.5143342   1.93573856 -3.07475281 -3.5244019   2.04414392  2.51948404
  2.39565015 -3.59928584 -3.27535367  1.9520036   2.

In [23]:
from numpy import trunc
vectorQ15 = vector_np * 2**15
# vectorQ15trunc = trunc(vectorQ15)
vectorQ15trunc = vectorQ15.astype(np.int32)

print(vectorQ15trunc)


[  54933  -50382  -75474  -53673   60663   12413  -10772   -7340  -20269
   16064  -14066  -13817    8685  -10113   -3144    4075   -4594    5321
  -12647   -8063   66066  -69865  -69534  -70804   72916   54063  -60324
  -59830  -53378   68752   -1571   -6276    5702   16456   -8790  -60862
   56589   69798   58197  -52280 -120504  108703  115628   95384 -123365
 -129741   95475  110671   98048 -139216 -125848   95522  108256   81343
 -137254 -124255   76410  106364   70798 -126158 -127826   70079   90550
   68222 -130677 -113711   55399   67613   56376 -117466 -101176   58320
   82389   63430 -100753 -115487   66982   82558   78500 -117941 -107326
   63963   65894   65660 -114810  -89238   44866   65651   59814 -102925
  -94928   45014   62425   60562  -96875  -78599   52307   69584   62395
  -93306  -82340   61712   74733   61414  -83816  -79047   54269   49906
   58515  -71592  -72178   53838   55633   41745  -60223  -59872   57286
   69733   58019  -71409  -71348   60486   56600   

In [29]:
# Convert to two's complement and print in 19-bit binary format
vector_q15_19bit = []
with open('../mem/param.mem', 'w') as mem_file:
    for value in vectorQ15trunc:
        if value < 0:
            # Convert negative value to two's complement
            value = (1 << 19) + value
        # Write the 19-bit binary value to the .mem file
        mem_file.write(format(value, '019b') + '\n')
